# Image Search with CLIP + FAISS
This notebook builds an image-search AI using CLIP embeddings (openai/clip-vit-base-patch32) and FAISS for fast nearest-neighbor search. Run cells in order. Upload or mount a folder of images, compute embeddings, build a FAISS index, then search by image or text.


In [ ]:
# Section 1 — Install dependencies
!pip install -q transformers==4.30.0 ftfy regex tqdm sentencepiece
!pip install -q faiss-cpu pillow matplotlib gradio
!pip install -q accelerate

# Show versions
import sys, pkgutil
import torch
print('torch', torch.__version__)
import transformers
print('transformers', transformers.__version__)
import faiss
print('faiss', faiss.__version__ if hasattr(faiss, '__version__') else 'faiss-cpu')

In [ ]:
# Section 2 — Imports and device
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from transformers import CLIPProcessor, CLIPModel
import faiss
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Section 3 — Mount Google Drive (optional) and create folders
# Uncomment to mount Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Project folders (in Colab use /content/project_images or under /content/drive/MyDrive/...)
base_dir = '/content/image_search_project'
folders = ['images','embeddings','index','metadata']
for f in folders:
    os.makedirs(os.path.join(base_dir, f), exist_ok=True)
print('Base dir:', base_dir)
print('Folders created:', folders)


In [ ]:
# Section 4 — Upload or fetch images
# Option A: upload via Colab UI
from google.colab import files
print('Use files.upload() to upload images to Colab directly (uncomment to use).')
# uploaded = files.upload()
# for name in uploaded:
#     open(os.path.join(base_dir,'images', name),'wb').write(uploaded[name])

# Option B: copy from Drive (after mount)
# !cp -r /content/drive/MyDrive/some_image_folder/* {os.path.join(base_dir,'images')}

# Option C: download a small public dataset (example)
# !git clone https://github.com/fastai/imagenette.git && cp -r imagenette/* {os.path.join(base_dir,'images')}

# After populating, list images
image_dir = os.path.join(base_dir,'images')
imgs = [os.path.join(image_dir,f) for f in os.listdir(image_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print('Found', len(imgs), 'images')


In [ ]:
# Section 5 — Image preprocessing & batched loader
from PIL import Image

def open_image(path, size=224):
    img = Image.open(path).convert('RGB')
    return img

from math import ceil

def batch_generator(paths, batch_size=32):
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        images = [open_image(p) for p in batch_paths]
        yield batch_paths, images

# Quick sample: show first image if exists
if len(imgs) > 0:
    display(open_image(imgs[0]))


In [ ]:
# Section 6 — Load CLIP model & processor
model_name = 'openai/clip-vit-base-patch32'
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()
print('Loaded CLIP model:', model_name)


In [ ]:
# Section 7 — Compute image embeddings (batched)
import numpy as np

def compute_image_embeddings(paths, batch_size=32):
    all_embs = []
    ids = []
    for batch_paths, images in tqdm(batch_generator(paths, batch_size=batch_size), desc='Embedding'):
        inputs = processor(images=images, return_tensors='pt').to(device)
        with torch.no_grad():
            image_emb = model.get_image_features(**inputs)
        # Move to CPU and convert
        image_emb = image_emb.cpu().numpy().astype('float32')
        all_embs.append(image_emb)
        ids.extend(batch_paths)
    if len(all_embs) == 0:
        return np.zeros((0, model.config.projection_dim), dtype='float32'), []
    embs = np.vstack(all_embs)
    return embs, ids

# Example run (will do nothing if no images)
if len(imgs) > 0:
    embs, ids = compute_image_embeddings(imgs[:100], batch_size=16)
    print('Embeddings shape:', embs.shape)
else:
    embs = np.zeros((0, model.config.projection_dim), dtype='float32')
    ids = []


In [ ]:
# Section 8 — Normalize embeddings and persist
from numpy.linalg import norm

def l2_normalize(x, axis=1, eps=1e-10):
    norms = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (norms + eps)

if embs.shape[0] > 0:
    embs = l2_normalize(embs)
    np.save(os.path.join(base_dir,'embeddings','image_embs.npy'), embs)
    # Save metadata
    import json
    with open(os.path.join(base_dir,'metadata','image_ids.json'),'w',encoding='utf-8') as f:
        json.dump(ids, f, ensure_ascii=False, indent=2)
    print('Saved embeddings and metadata')
else:
    print('No embeddings to save')


In [ ]:
# Section 9 — Build FAISS index and save/load
import faiss

dim = embs.shape[1] if embs.shape[0]>0 else model.config.projection_dim
index = None
if embs.shape[0] > 0:
    # Using IndexFlatIP (inner product) since vectors are normalized => cosine similarity
    index = faiss.IndexFlatIP(dim)
    index.add(embs)
    print('Added', index.ntotal, 'vectors to FAISS index')
    faiss.write_index(index, os.path.join(base_dir,'index','faiss_index.idx'))
    print('Saved FAISS index')
else:
    print('No vectors to index')

# To load later:
# index = faiss.read_index(os.path.join(base_dir,'index','faiss_index.idx'))


In [ ]:
# Section 10 — Query embeddings (text and image) and search

def get_text_embedding(texts):
    inputs = processor(text=texts, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        t_emb = model.get_text_features(**inputs)
    t_emb = t_emb.cpu().numpy().astype('float32')
    t_emb = l2_normalize(t_emb)
    return t_emb

def search_faiss(query_vec, k=10):
    if index is None:
        raise RuntimeError('Index not built')
    query_vec = query_vec.astype('float32')
    # ensure shape (n, d)
    if query_vec.ndim == 1:
        query_vec = query_vec.reshape(1, -1)
    D, I = index.search(query_vec, k)
    return D, I

# Example text search
if embs.shape[0] > 0:
    q = get_text_embedding(['living room modern sofa'])[0]
    D,I = search_faiss(q, k=5)
    print('Scores:', D[0])
    print('Indices:', I[0])


In [ ]:
# Section 11 — Display results helper
import math

def display_results_from_indices(indices, scores, ids, cols=5, thumb_size=(224,224)):
    # indices: list of indices (ints) or array shape (k,)
    fig_cols = cols
    fig_rows = math.ceil(len(indices)/fig_cols)
    plt.figure(figsize=(4*fig_cols,3*fig_rows))
    for i, idx in enumerate(indices):
        path = ids[idx]
        score = float(scores[i])
        img = open_image(path)
        plt.subplot(fig_rows, fig_cols, i+1)
        plt.imshow(img)
        plt.title(f"{os.path.basename(path)}\nscore: {score:.3f}")
        plt.axis('off')
    plt.tight_layout()

# If we did a text query above and have I and D, display
if embs.shape[0] > 0:
    display_results_from_indices(I[0], D[0], ids, cols=5)


In [ ]:
# Section 12 — Interactive demo with Gradio
import gradio as gr

# helper that accepts either image or text

def run_search(image=None, text=None, top_k=5):
    if text and text.strip() != '':
        q = get_text_embedding([text])[0]
    elif image is not None:
        # image is a PIL.Image
        inputs = processor(images=[image], return_tensors='pt').to(device)
        with torch.no_grad():
            q = model.get_image_features(**inputs).cpu().numpy().astype('float32')
        q = l2_normalize(q)[0]
    else:
        return ['No query provided'], []
    D,I = search_faiss(q, k=top_k)
    results = [ids[i] for i in I[0]]
    scores = [float(s) for s in D[0]]
    thumbs = []
    for p in results:
        try:
            img = open_image(p)
            thumbs.append(img)
        except:
            thumbs.append(None)
    return thumbs, results, scores

iface = gr.Interface(
    fn=run_search,
    inputs=[gr.Image(type='pil', label='Query Image (optional)'), gr.Textbox(label='Text Query (optional)'), gr.Slider(minimum=1, maximum=20, step=1, label='Top K', value=5)],
    outputs=[gr.Gallery(label='Top Matches').style(grid=5), gr.JSON(label='Paths'), gr.JSON(label='Scores')],
    examples=[[]],
    title='Image Search (CLIP + FAISS)'
)

# Launch — in Colab this will produce a public link
iface.launch(share=True)


# Section 13 — Evaluation and tips

# Precision@k example (requires labeled ground truth)
# def precision_at_k(relevant_set, retrieved_list, k):
#     retrieved_k = retrieved_list[:k]
#     hits = sum([1 for r in retrieved_k if r in relevant_set])
#     return hits / k

# Tips:
# - For large datasets (>100k), use FAISS IVF or HNSW for speed and memory efficiency.
# - Compute embeddings in batches (and save periodically) to avoid OOM.
# - Use GPU for faster embedding computation.
# - Keep an index-to-path mapping (JSON or CSV) for retrieval.

# End of notebook — run cells top->bottom. Upload images to '/content/image_search_project/images' or mount Drive and point the base_dir accordingly.
